In [37]:
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats

In [18]:
df_fixtures = pd.read_csv("wc_2026_fixtures_clean.csv")
df_teams = pd.read_csv("wc_2026_teams.csv")
df_editions = pd.read_csv("wc_all_editions_clean.csv")
df_matches = pd.read_csv("wc_all_matches_clean.csv")
df_scorers = pd.read_csv("wc_top_scorers_clean.csv")
dfs = {"fixtures":df_fixtures, "teams":df_teams, "editions":df_editions, "matches":df_matches, "scorers":df_scorers}

In [19]:
for name, df in dfs.items():
    print(f"Columns in {name}: {df.columns.to_list()}\n")

Columns in fixtures: ['group', 'stage', 'team1', 'team2', 'venue', 'city', 'country', 'date', 'kickoff_et', 'team1_confederation', 'team1_fifa_rank', 'team1_coach', 'team2_confederation', 'team2_fifa_rank', 'team2_coach', 'manually_corrected']

Columns in teams: ['team', 'group', 'confederation', 'fifa_rank', 'coach', 'best_wc_result', 'debut_2026']

Columns in editions: ['edition', 'year', 'host', 'champion', 'runner_up', 'third_place', 'fourth_place', 'teams', 'matches', 'goals', 'goals_per_match', 'attendance', 'top_scorer', 'top_scorer_country', 'top_scorer_goals', 'start_date', 'end_date', 'final_city', 'host_won', 'format', 'format_raw', 'host_won_raw', 'champion_canon', 'runner_up_canon', 'third_place_canon', 'fourth_place_canon', 'top_scorer_country_canon', 'host_canon']

Columns in matches: ['year', 'stage', 'team1', 'score1', 'score2', 'team2', 'venue', 'city', 'country', 'date', 'notes', 'manually_corrected', 'team1_canon', 'team2_canon', 'country_canon', 'stage_clean', 'sta

## Data Analysis

### part 1

In [20]:
print(df_editions["edition"].count())

22


In [21]:
df_fixtures.shape[0]

104

In [22]:
total_goals = df_matches["total_goals"].sum() # not accurate since only 184 matches are included in the dataset, but there have been 900 matches in total.
print(total_goals)

647


In [23]:
avg_goals = total_goals / df_matches.shape[0]
print(f"{avg_goals:.2f}")

3.52


In [24]:
df_editions["attendance"].sum() / df_editions["attendance"].count()

np.float64(1981512.9090909092)

In [25]:
df_editions["host"].nunique()

18

In [26]:
teams = set(df_matches["team1"]) | set(df_matches["team2"])
print(len(teams)) # not accurate since only 184 matches are included in the dataset, but there have been 900 matches in total.

54


### part 2

In [27]:
goals_per_year = df_editions[["year","goals", "edition"]]
fig = px.line(
    goals_per_year, 
    x="year",
    y="goals",
    title="Total Goals per Year",
    markers=True
)

fig.show()

In [28]:
attendance_per_year = df_editions[["year","attendance", "edition"]]

fig = px.line(
    attendance_per_year, 
    x="year",
    y="attendance",
    title="Total Attendance per Year",
    markers=True
)

fig.show()

In [29]:
goals_per_match_per_year = df_editions[["year","goals_per_match"]]
fig = px.line(
    goals_per_match_per_year, 
    x="year",
    y="goals_per_match",
    title="Total (Goals per match) per Year",
    markers=True
)

fig.show()

In [30]:
teams_per_year = df_editions[["year","teams"]]
fig = px.line(
    teams_per_year, 
    x="year",
    y="teams",
    title="Total Teams per Year",
    markers=True
)

fig.show()

In [38]:
fig = make_subplots(rows=1, cols=2)

fig1 = px.box(
    attendance_per_year,
    points="all",
    y="attendance",
    hover_data=["year", "edition"],
    title="Total Attendance per Year",
)

fig2 = px.box(
    goals_per_year,
    points="all",
    y="goals",
    hover_data=["year", "edition"],
    title="Total Goals per Year",
)

for trace in fig1.data:
    fig.add_trace(trace, row=1, col=1)

for trace in fig2.data:
    fig.add_trace(trace, row=1, col=2)

fig.update_layout(showlegend=False)

fig.show()
# no outlires!

In [40]:
df_editions[
    ["year", "teams", "matches", "goals", "goals_per_match"]
].sort_values("goals_per_match", ascending=False).head()

,year,teams,matches,goals,goals_per_match
4,1954,16,26,140,5.38
2,1938,15,18,84,4.67
1,1934,16,17,70,4.12
3,1950,13,22,88,4.00
0,1930,13,18,70,3.89


In [41]:
df_editions["decade"] = (
    df_editions["year"] // 10 * 10
)

decade_stats = (
    df_editions.groupby("decade")
    .agg(
        editions=("year", "count"),
        avg_goals_per_match=("goals_per_match", "mean"),
        total_goals=("goals", "sum"),
        total_matches=("matches", "sum")
    )
    .reset_index()
)

decade_stats

,decade,editions,avg_goals_per_match,total_goals,total_matches
0,1930,3,4.226667,224,53
1,1950,3,4.326667,354,83
2,1960,2,2.780000,178,64
3,1970,3,2.733333,294,108
4,1980,2,2.675000,278,104
5,1990,3,2.530000,427,168
6,2000,2,2.410000,308,128
7,2010,3,2.526667,485,192
8,2020,1,2.690000,172,64


In [42]:
decade_stats.sort_values(
    "avg_goals_per_match",
    ascending=False
)

,decade,editions,avg_goals_per_match,total_goals,total_matches
1,1950,3,4.326667,354,83
0,1930,3,4.226667,224,53
2,1960,2,2.780000,178,64
3,1970,3,2.733333,294,108
8,2020,1,2.690000,172,64
4,1980,2,2.675000,278,104
5,1990,3,2.530000,427,168
7,2010,3,2.526667,485,192
6,2000,2,2.410000,308,128


In [43]:
fig = px.line(
    df_editions.sort_values("year"),
    x="year",
    y="goals_per_match",
    markers=True,
    title="Goals per Match Across World Cup History",
    labels={
        "year": "World Cup Year",
        "goals_per_match": "Average Goals per Match"
    }
)

fig.show()

### part 3